In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-6: GRPO refinements

## RL for LLMs, Part 1 (PPO, GRPO and their refinements), session 6 of 14

## What this session teaches

Plain GRPO samples a group of answers to one question and gives each answer one number, its advantage: how far its reward is above the group's average, divided by the group's standard deviation.

Every token of that answer is pushed up or down by that same number.

Three more choices surround this. (1) The loss is averaged over each answer's tokens, so a token in a long answer counts less than a token in a short one.

(2) A clip stops the push on a token once its probability has changed by a set fraction. (3) A penalty, the KL term, holds the model near its starting point.

Later papers changed these choices, which groups are used, and how answers cut off by the length limit are scored.

The changes do not all stand on the same footing.

Some are arithmetic: how much a token counts under each averaging rule. Some are arguments that an old choice does harm. Some are design choices: choices with no proof, only a test in one setting.

None touches what GRPO gives up: every token of an answer gets the same number.

The question to carry away is what kind of reason stands behind each change.

**By the end you can**
- say which parts of GRPO later papers change, and why
- tell which changes rest on arithmetic, which on arguments, and which are design choices
- say what GRPO still gives up after all of them

## Setting and words

> **Words the first sections use. Two are bridges: an answer is a completion, and AIME 2024 is the benchmark from RLLM-4.**

- **completion:** the whole text the model writes for one prompt
  - the opening said answer; from here on, completion
- **AIME 2024:** competition math with exact numeric answers (recall from RLLM-4)
  - a score is the share solved
- **reported number:** the score a paper states for its own run
  - a **reimplementation** is another group's run of a method
- **cut-off:** a completion stopped by the length limit
- three papers whose changes follow:
  - **DAPO:** an open training recipe that reports 50 on AIME 2024
  - **Dr. GRPO:** argues that the averaging and the standard deviation bias the loss
  - **GSPO:** moves the ratio and the clip from tokens to whole completions

## How we will teach it: the map

> **Five sections in one chain: why look beyond plain GRPO, what its update is made of, what each change does to one part, what is still missing, and where Part 2 goes.**

<img src="images/rllm6v2/map_all.png" width=70% alt="Chain of five sections: Reasons, Objective, Refinements (five groups of changes inside), Credit, Hand-off, each with its question.">

- Sections: **Reasons, Objective, Refinements, Credit, Hand-off**
- A **knob** is the part of the update that a change adjusts: the averaging, the clip, the KL term
- Each change gets three lines: **the knob**, **the change**, and **believe it?**, the kind of reason behind it
- Labels: a **fact** comes from a paper's own report
  - **our interpretation** (a claim this lecture reasons out from facts) or a **judgment** is this lecture's

## Reasons: two reasons to look beyond plain GRPO

**You are here:** Reasons (section 1 of 5)

<img src="images/rllm6v2/strip_reasons.png" width=62% alt="Strip of the deck's sections with the current one (reasons) highlighted.">

> **Neither reason is proof alone. Together they say where to look; the later evidence says what helped.**

<img src="images/rllm6v2/reasons.png" width=42% style="float:right; margin:0 0 6px 18px" alt="Bars of AIME 2024 scores: 30 for DAPO's own plain GRPO run, 47 reported by DeepSeek, 50 for the DAPO recipe.">

- **Reason 1, a reported gap (fact):** DAPO's own plain GRPO run scored 30 on AIME 2024
  - DeepSeek reported 47 for DeepSeek-R1-Zero-Qwen-32B
    - the R1-Zero recipe on a 32B Qwen base, a smaller model than RLLM-4's R1-Zero (77.9, on V3-Base)
  - **caveat:** the 30 is a reimplementation, and DeepSeek's recipe is not public, so undisclosed details may explain the gap
- **Reason 2, a hypothesis that the objective can be improved (our interpretation):** each possible improvement is a claim, tested later
  - length averaging, advantage scale, rare-token clip, wasted groups, cut-offs
- a score here is avg@32: the share solved, averaged over 32 repeats of the test set
  - not comparable with RLLM-4's 77.9

## Objective: the parts of the update, named first

**You are here:** Objective (section 2 of 5). To change an objective we must name its parts (recalled from C2 and RLLM-4).

<img src="images/rllm6v2/strip_objective.png" width=62% alt="Strip of the deck's sections with the current one (objective) highlighted.">

> **Plain GRPO gives each completion one advantage and gives that same number to every token. The next slide's formula combines it with the ratio, the clip, the KL term and two averages.**

<img src="images/rllm6v2/anatomy.png" width=50% style="float:right; margin:0 0 6px 18px" alt="One completion's tokens, a single advantage reaching every token, then ratio, clip and KL term per token, and the two averages.">

- **group:** G completions sampled for one prompt (C2)
- **advantage:** reward minus the group mean, divided by the group's standard deviation (C2)
- **ratio** (importance ratio): the token's new probability over its old one (B4)
- **clip:** limits the ratio to $1 \pm \epsilon$ (B4)
- **KL term:** penalizes the policy's distance from the reference model, scaled by the KL coefficient $\beta$ (C1)
- **credit:** which token earned how much of the reward (RLLM-5)
  - plain GRPO gives every token the same advantage, so no separate credit

## Objective: the same parts, in one formula

> **Per token: PPO's clipped term using the completion's one advantage, minus a KL term; averaged over the tokens of the completion, then over the group.**

$$
\frac{1}{G} \sum_{i=1}^{G} \frac{1}{|y_i|} \sum_{t} \Bigl[ \operatorname{min}\bigl( r_{i,t}\, A_i,\; \operatorname{clip}(r_{i,t},\, 1 - \epsilon,\, 1 + \epsilon)\, A_i \bigr) - \beta \, \text{KL}_{i,t} \Bigr]
$$

$$
A_i = \frac{R_i - \text{mean}(R)}{\text{std}(R)}
$$

- $i$ numbers the completions, $t$ the tokens, $R_i$ is completion $i$'s reward (completion first, as in C2)
- $y_i$ is completion $i$, a sequence of tokens
  - $|y_i|$ is its length in tokens
- $\text{KL}_{i,t}$ is GRPO's per-token KL estimate (C2)
- **what changes, from what:** the policy's parameters (the model being trained), using a group of its own completions and their verifier rewards (RLLM-4)

## Refinements: averaging, how the loss is averaged

**You are here:** Refinements, averaging (1 of 5 groups)

<img src="images/rllm6v2/strip_averaging.png" width=62% alt="Strip of the deck's sections with the current one (averaging) highlighted.">

> **GRPO divides by the completion's own length first, so each token of a long completion counts less than each token of a short one.**

<img src="images/rllm6v2/share.png" width=34% style="float:right; margin:0 0 6px 18px" alt="Bars of each token's share of the loss for a 10-token and a 100-token completion under plain GRPO and token-level averaging.">

- **the knob:** how much each token's term counts in the loss: its **share of the loss**, a fixed factor, not the advantage
- **the change:** count all tokens of the group alike (DAPO)
  - Dr. GRPO's rival rule, a constant divisor, comes with its other change on the next slide
- **believe it?** The sizes are arithmetic: share = 1/(G x length)
  - G = 2 completions of 10 and 100 tokens
    - per token of the short one: 1/(2 x 10) = 0.0500
    - per token of the long one: 1/(2 x 100) = 0.0050: 10 to 1
  - counting all tokens alike gives each 1/110
- DAPO's argument: good long reasoning is learned slowly, long gibberish is penalized little (ablation: RLLM-6A)
- the open length question: Dr. GRPO argues that part of the length growth is this averaging (an argument, in RLLM-6A)

*Credit gap untouched: under this change GRPO still gives each token the same advantage.*

## Refinements: advantage scale, whether to divide by the standard deviation

**You are here:** Refinements, advantage scale (2 of 5)

<img src="images/rllm6v2/strip_advscale.png" width=62% alt="Strip of the deck's sections with the current one (advscale) highlighted.">

> **Dr. GRPO argues that dividing by the group standard deviation scales up the advantage of almost-all-wrong and almost-all-right questions, relative to the undivided advantage.**

<img src="images/rllm6v2/advscale.png" width=38% style="float:right; margin:0 0 6px 18px" alt="Bars of the advantage of a correct completion in a lone-success group and a balanced group, divided and not divided by the standard deviation.">

- **the knob:** whether to divide the reward gap by the group standard deviation
- **why GRPO divides:** it puts questions with different reward spreads on one scale (C2)
- **the change:** drop the division (Dr. GRPO, the paper that also proposed a constant divisor in place of length averaging)
  - DAPO and GSPO keep it: an open design choice
- **believe it?** The amplification is arithmetic: 2.47 against 0.88 for a lone success, 0.94 against 0.50 when balanced
  - that it harms is an argument
- **Our interpretation:** the argument assumes both questions are equally informative
  - a lone success is rarer, so it could deserve more

*Credit gap untouched: under this change GRPO still gives each token the same advantage.*

## Refinements: clip, a higher upper edge

**You are here:** Refinements, clip (3 of 5)

<img src="images/rllm6v2/strip_clip.png" width=62% alt="Strip of the deck's sections with the current one (clip) highlighted.">

> **From RLLM-5: the clip is a heuristic brake, not a bound on the KL, so its shape is a design choice. The upper edge matters only for a positive advantage.**
<!-- seed-pay: clip-heuristic -->

<img src="images/rllm6v2/cap.png" width=32% style="float:right; margin:0 0 6px 18px" alt="Bars of the room a token's probability has to grow before the clip acts, for four starting probabilities and two upper edges.">

- **the knob:** the width of the clip on the ratio (new over old probability)
- **the change:** a higher upper edge, 0.28 against 0.2 (DAPO), the lower edge kept at 0.2
- the clip removes that token's gradient: not a hard cap (RLLM-5)
- the clip acts only when a batch is reused: DAPO does, 16 updates per rollout step
- **believe it?** The sizes are arithmetic: the clip acts at 1.2 times the old probability
  - a token at 0.01 can rise by 0.002, one at 0.5 by 0.1
  - at 0.9 the room is 0.1 either way (a probability cannot pass 1)
- **DAPO's argument:** with the old edge a rare token cannot rise much, so rare tokens, the useful exploration, stay rare
  - our interpretation: raising them matters most early in training (not checked against the paper's wording)
- **our interpretation:** the mechanism is the paper's explanation, not a measurement
  - the gain was measured by an ablation, DAPO's run with one change removed (RLLM-6A)

*Credit gap untouched: under this change GRPO still gives each token the same advantage.*

## Refinements: clip, entropy collapse

> **DAPO observed entropy collapse: the next-token distribution concentrates on a few tokens. Its proposed cause, a relative cap that lets rare tokens grow only a little per update, is our interpretation.**

<img src="images/rllm6v2/entropy.png" width=48% style="float:right; margin:0 0 6px 18px" alt="Two bar charts of a next-token distribution: spread out before entropy collapse, concentrated on one token after.">

- **entropy** measures how spread the next-token distribution is; 0 means one token
- **entropy collapse:** the distribution concentrates on a few tokens, so a model's samples differ less from each other
- the mechanism (rare tokens explore and grow slowly) is our interpretation
- the collapse itself was observed by the paper
- **why collapse hurts:** exploration needs a rare but possibly important token to stay possible
  - collapse is exactly the loss of that probability, so the model stops trying alternatives
- the paper keeps the lower epsilon at 0.2 (floor 0.8)
  - a larger one lowers the floor and lets a rare token's probability fall toward zero: the collapse we want to avoid
- **risk:** tuned in one setting

## Refinements: clip, one ratio per completion

**You are here:** Refinements, clip (3 of 5), the second of its two changes: GSPO's completion ratio

<img src="images/rllm6v2/strip_clip.png" width=62% alt="Strip of the deck's sections with the current one (clip) highlighted.">

> **GSPO defines the ratio, and its clip, on the whole completion instead of on each token.**

<img src="images/rllm6v2/ratios.png" width=32% style="float:right; margin:0 0 6px 18px" alt="Twelve token ratios against the clip band with four outside it, and the geometric mean of all twelve inside the band.">

- **the knob:** the unit the ratio and clip act on: a token, or a completion
- **the change:** one **completion ratio**, the geometric mean of the token ratios (the paper's "sequence ratio")
  - clipped once: every token gets the same clip decision
- **why:** the advantage belongs to the whole completion, but a single token's ratio is noisy: one noisy token can trigger the clip and silence itself
  - the mean averages that noise out
- **believe it?** The paper's argument, tested in its own models
  - the figure is a mechanism, not a measurement (12 ratios give 1.00, inside the band)
  - formula and real clip ranges: RLLM-6A

*Credit gap untouched: under this change GRPO still gives each token the same advantage.*

## Refinements: group filtering and cut-offs, do not waste groups

**You are here:** Refinements, groups and cut-offs (4 of 5)

<img src="images/rllm6v2/strip_groups.png" width=62% alt="Strip of the deck's sections with the current one (groups) highlighted.">

> **From RLLM-4: if every completion in a group earns the same reward, all advantages are 0, so the update moves no parameter and the sampling work is wasted.**
<!-- seed-pay: all-equal-group -->

<img src="images/rllm6v2/allequal.png" width=40% style="float:right; margin:0 0 6px 18px" alt="Curve of the chance that a group of eight completions is all-equal against the success rate, with points at 0.5 and 0.9.">

- **setting:** from here to the end of the refinements, the changes are DAPO's, for long-reasoning training on math (RLLM-4)
- **the knob:** which groups enter the update
- **the change: dynamic sampling** (DAPO): keep a prompt only if its group has a success and a failure
  - sample until the batch is full
- **believe it?** The waste is arithmetic
  - with $q$ the chance that one completion of this prompt is correct, an all-equal group of 8 has chance $q^8 + (1-q)^8$
  - 0.008 at $q = 0.5$, 0.43 at $q = 0.9$; the benefit is DAPO's ablation
- as the policy improves, more prompts reach the right-hand end (our interpretation)
- the paper notes more generation per step, but fewer steps

*Credit gap untouched: under this change GRPO still gives each token the same advantage.*

## Refinements: group filtering and cut-offs, do not punish cut-offs unfairly

> **A cut-off completion may get a punishing reward even if its reasoning was sound; the DAPO paper argues this adds noise.**

<img src="images/rllm6v2/softpen.png" width=44% style="float:right; margin:0 0 6px 18px" alt="Penalty against completion length: zero up to 16,384 tokens, then a ramp to minus one at the 20,480 limit.">

- **setting:** still DAPO's long-reasoning setting, where completions run to thousands of tokens
- **the knob:** the reward of a completion stopped by the length limit
- **the change: overlong filtering** (mask its loss, so it gives no gradient) or **soft overlong punishment** (a penalty that grows only near the limit)
- **believe it?** Design choices; the evidence is DAPO's ablation (RLLM-6A), not a proof
- under plain GRPO a cut-off has no final answer, so it fails the accuracy check, and usually the format check
- the paper's soft setting is the figure: no penalty to 16,384 tokens, then linear to -1 over 4,096

*Credit gap untouched: under this change GRPO still gives each token the same advantage.*

## Refinements: KL term, drop it?

**You are here:** Refinements, KL term (5 of 5)

<img src="images/rllm6v2/strip_kl.png" width=62% alt="Strip of the deck's sections with the current one (kl) highlighted.">

> **Both papers set the KL coefficient to zero, for different reasons. Judgment: drop it with a verifier; think twice with a learned reward model.**

- **the knob:** the KL term, which penalizes distance from the reference model, the frozen copy the policy started from (C1, RLLM-5)
- **why $\beta > 0$ was sensible:** in RLHF the goal is to align the model without diverging far from the initial model (DAPO's words)
  - our interpretation: this also limits reward hacking
    - a learned reward model gets exploited, which DAPO notes separately as its reason for a rule-based reward
  - **why it misfits here:** a long-reasoning model can diverge a lot from its start, so the restriction is not necessary (DAPO's words)
- **the change:** set the KL coefficient $\beta = 0$ (DAPO, Dr. GRPO)
- **believe it?** Two arguments, no ablation number: a design choice
  - **DAPO:** a long-reasoning model should move far from its start
  - **Dr. GRPO:** with a verifier (RLVR, RLLM-4) there is no learned reward model to exploit
- benefit: no reference model in memory; risk: drift the reward does not forbid (RLLM-6A)

*Credit gap untouched: under this change GRPO still gives each token the same advantage.*

## Refinements: the seven changes by kind of reason

> **A menu, not a recipe, except one bundle: DAPO's was run together and answers reason 1. The sizes in four changes are plain arithmetic; none of those sizes shows a change helps.**

<img src="images/rllm6v2/kinds.png" width=66% alt="Table of the seven changes with filled squares for arithmetic (a size), argument, design choice and the paper's test.">

- **combine:** the DAPO bundle (token-level averaging, clip higher, group filtering, cut-off handling, no KL)
  - dropping the std fits with it, which DAPO leaves open
- **conflict:** two rules for one knob
  - Dr. GRPO's constant divisor against token-level counting
  - GSPO's completion ratio against clip higher: one clips completions, the other tokens
- the baseline is untouched by every change; the next section returns to it
  - recall from RLLM-5: the critic's value in PPO, the group mean in GRPO

## Credit: what GRPO gives up

**You are here:** Credit (section 4 of 5). The seven changes leave credit assignment alone, and credit assignment is where PPO and GRPO differ.

<img src="images/rllm6v2/strip_credit.png" width=62% alt="Strip of the deck's sections with the current one (credit) highlighted.">

> **From RLLM-5: the critic had three jobs, and GRPO has no critic. Its group mean does the baseline job. Nothing does credit assignment: every token gets one advantage, so GRPO cannot single out the token that mattered.**
<!-- seed-pay: critic-two-roles -->

<img src="images/rllm6v2/credit.png" width=42% style="float:right; margin:0 0 6px 18px" alt="Two bar charts: PPO gives every token its own advantage, GRPO gives every token the same advantage.">

- the group mean does the baseline job, and no change touches it
- none of the seven changes restores credit assignment
  - a token that wasted a step still shares the completion's advantage
- in return GRPO needs no critic, no second large network (RLLM-5)
- next: Part 1 ends here, and Part 2 asks where methods can differ

## Hand-off: Part 1 ends here, Part 2 begins

**You are here:** Hand-off (section 5 of 5). The end of Part 1, and the question Part 2 asks.

<img src="images/rllm6v2/strip_handoff.png" width=62% alt="Strip of the deck's sections with the current one (handoff) highlighted.">

> **Part 1 asked how to learn from a reward. Part 2 asks what changes when the samples or the weights come from somewhere else. Three axes organize it.**

<img src="images/rllm6v2/axes.png" width=46% style="float:right; margin:0 0 6px 18px" alt="Three rows for the three axes with their answers; PPO and GRPO highlighted, the same on axes 1 and 2 and different on axis 3.">

- Part 1 (RLLM-1 to RLLM-6): every method sampled from the model itself and took its weights from a reward
- a **weight** is the number a method sets for a token from its reward or imitation target (GRPO's: the advantage)
- **1. whose samples:** the model's own (a copy within a few updates is nearly the same), or another's (RLLM-7)
- **2. where the weights come from:** something to imitate (a dataset's token, a teacher), or a reward (RLLM-8)
- **3. how fine the weights are:** per completion, per sampled token, or a distribution per token (RLLM-8)
- PPO and GRPO answer 1 and 2 alike, and differ on 3: the credit gap
- same axes as RLLM-1, reworded: RLLM-1 named axis 2 by who provides the weights, axis 3 by how fine the credit is

## Hand-off: choosing a method after Part 1

> **What you have decides the reward source. This covers Part 1 options only; the full rule, with Part 2 options, comes in RLLM-13.**

<img src="images/rllm6v2/choice.png" width=58% alt="What you have, to reward source, to PPO or GRPO: program check gives RLVR, people give RLHF, an AI rater gives RLAIF.">

- **a program checks the answer:** RLVR, reinforcement learning with verifiable rewards (RLLM-4)
- **only people can rate the outputs:** collect comparisons, train a reward model, run RLHF (RLLM-2)
- **an AI model can rate them, guided by written principles:** RLAIF (RLLM-3)
- PPO or GRPO, in practice (judgment): RLVR recipes often start from GRPO
  - papers on repaired critics (VAPO, VC-PPO) argue PPO can win
- next: methods that change where the samples or the weights come from

# What we established, and what it sets up

- **Established**
  - two reasons to look beyond plain GRPO, neither proof alone: a reported gap, and a hypothesis that the objective can be improved
  - seven changes, one per part of the update
  - the sizes in four (averaging, advantage scale, clip higher, empty groups) are plain arithmetic
  - whether a change helps is not arithmetic
  - that it helps is an argument or a design choice tested in one setting: options, not a ranking
  - **DAPO:** token-level averaging, higher clip, dynamic sampling, cut-off handling, no KL
  - **Dr. GRPO:** no length or std division, no KL
  - **GSPO:** one ratio per completion
- **Sets up**
  - **Part 1 is complete:** rewards from people, models or programs, and PPO or GRPO to use them
  - **the limitation Part 2 starts from:** one advantage per completion, shared by every token, from the current policy's own samples
  - **Part 2, from RLLM-7 (RFT alternative methods):** vary the three axes, beginning with whose samples
  - **which method to pick:** the Part 1 rule is on the previous slide
  - the full rule, with Part 2 options, is in RLLM-13

## Aside: where the detail is (RLLM-6A)

- the optional appendix RLLM-6A holds: Dr. GRPO's argument about wrong answers drifting longer
- GSPO's completion ratio in detail and its real clip ranges
- what dropping the KL term risks
- the DAPO ablation, step by step, and what it does not show
- papers (arXiv): DAPO 2503.14476; Dr. GRPO 2503.20783; GSPO 2507.18071